In [6]:
from rdkit import Chem
import torch
from protlig.helper import load_dataset

In [7]:
esol_data = load_dataset()

/Users/tungvuduc/projects/protlig/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ELEMENTS = ["C", "N", "O", "F", "S", "Cl", "Br", "I", "P"]
N_FEATURES = len(ELEMENTS) + 1 + 2 # one-hot + other + degree + aromatic
DTYPE = torch.float64

def mol_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    rows = []
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        onehot = [float(sym == e) for e in ELEMENTS]
        onehot.append(float(sym not in ELEMENTS))
        rows.append(onehot + [atom.GetDegree(), atom.GetIsAromatic()])
    x = torch.tensor(rows, dtype=DTYPE)

    src, dst = [], []

    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        src += [i, j]
        dst += [j, i]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return x, edge_index, torch.tensor([y], dtype=DTYPE)

smiles = esol_data['SMILES'].iloc[0]
y = esol_data['label'].iloc[0]
mol_to_graph(smiles, y)

(tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 1.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 1.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 1.

In [74]:
def collate(graphs):

    xs, eis, ys, batch = [], [], [], []
    offset = 0

    for g_idx, (x, ei, y) in enumerate(graphs):
        n = x.size(0)
        xs.append(x)
        eis.append(ei + offset)
        ys.append(y)
        batch.append(torch.full((n, ), g_idx, dtype=torch.long))
        offset += n
    return (
        torch.cat(xs, dim=0),
        torch.cat(eis, dim=1),
        torch.cat(ys, dim=0),
        torch.cat(batch, dim=0),
    )
graphs = [mol_to_graph(s, y) for (s, y) in zip(esol_data['SMILES'], esol_data['label'])]
collate(graphs)

(tensor([[0., 0., 1.,  ..., 0., 1., 0.],
         [1., 0., 0.,  ..., 0., 2., 0.],
         [1., 0., 0.,  ..., 0., 3., 0.],
         ...,
         [1., 0., 0.,  ..., 0., 2., 1.],
         [1., 0., 0.,  ..., 0., 3., 1.],
         [0., 0., 0.,  ..., 0., 1., 0.]], dtype=torch.float64),
 tensor([[    0,     1,     1,  ..., 14990, 14989, 14982],
         [    1,     0,     2,  ..., 14989, 14982, 14989]]),
 tensor([-0.7700, -3.3000, -2.0600,  ..., -3.0910, -3.1800, -4.5220],
        dtype=torch.float64),
 tensor([   0,    0,    0,  ..., 1127, 1127, 1127]))

In [76]:
def global_add_pool(h, batch, n_graphs):
    out = torch.zeros(n_graphs, h.shape[1], dtype=h.dtype)
    return out.index_add_(0, batch, h)

xs, eis, ys, batch = collate(graphs)
global_add_pool(xs, batch, n_graphs=len(esol_data))

tensor([[20.,  1., 11.,  ...,  0., 68.,  6.],
        [12.,  1.,  2.,  ...,  0., 32., 11.],
        [10.,  0.,  1.,  ...,  0., 20.,  0.],
        ...,
        [ 6.,  0.,  2.,  ...,  0., 22.,  0.],
        [ 5.,  0.,  0.,  ...,  0.,  8.,  0.],
        [10.,  0.,  4.,  ...,  0., 38.,  6.]], dtype=torch.float64)

In [77]:
def propogate(x, edge_index):
    out = x.clone()
    src, dst = edge_index
    return out.index_add_(0, dst, x[src])
x, edge_index, y = mol_to_graph(smiles, y)
propogate(x, edge_index)

tensor([[ 1.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  3.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  6.,  0.],
        [ 3.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  8.,  0.],
        [ 2.,  0.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  7.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  7.,  0.],
        [ 3.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  8.,  0.],
        [ 2.,  0.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.,  0.],
        [ 2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  8.,  0.],
        [ 3.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 10.,  1.],
        [ 2.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  6.,  0.],
        [ 1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  3.,  0.],
        [ 4.,  0.,  

In [80]:
import json
from skfp.model_selection import scaffold_train_test_split

smiles = esol_data['SMILES']
tr_idx, te_idx = scaffold_train_test_split(smiles, test_size=0.2, return_indices=True)
tr_idx, te_idx = list(map(int, tr_idx)), list(map(int, te_idx))

assert set(tr_idx).isdisjoint(te_idx)
assert sorted(tr_idx + te_idx) == list(range(len(smiles)))

with open("../data/split_scaffold.json", "w") as fh:
    json.dump({"train": tr_idx, "test": te_idx}, fh)

In [87]:
import numpy as np

def iterate_minibatches(graphs, batch_size, shuffle=True, seed=0):
    idx = np.arange(len(graphs))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for start in range(0, len(graphs), batch_size):
        yield collate([graphs[i] for i in idx[start: start + batch_size]])

for xs, eis, ys, batch in iterate_minibatches(graphs, 20, shuffle=True):
    print(xs.shape, eis.shape, ys.shape, batch.shape)
    break

torch.Size([277, 12]) torch.Size([2, 574]) torch.Size([20]) torch.Size([277])
